# Predicting Email Reply Likelihood

This project predicts whether an outbound email is likely to receive a **Reply** or **No Reply**.

The project uses the **Enron Email Corpus** and compares two deep learning models:

- **LSTM** — used as the baseline model.
- **DistilBERT** — used as the advanced model.

The workflow includes data preparation, reply labeling, manual validation, model training, model evaluation, model comparison, and deployment using **Azure Machine Learning**.

## Step 1 — Environment Setup and Imports

This step prepares the notebook environment by installing and importing the libraries required for data processing, model training, evaluation, and Azure ML deployment.

In [1]:
# Install all libraries required for this project.
# Keeping them together makes the notebook easier to set up and reproduce.

%pip install -q datasets
%pip install -q openpyxl
%pip install -q scikit-learn
%pip install -q "transformers==4.46.3" "accelerate<1.0"
%pip install -q torch
%pip install -q "numpy<2"
%pip install -q azure-ai-ml azure-identity

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import all main libraries used throughout the project.
# Keeping imports together avoids repeating them in later sections.

# General data processing
import os
import re
import json
from datetime import timedelta
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# Dataset loading
from datasets import load_dataset

# Data splitting, validation and evaluation
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# LSTM model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# DistilBERT / PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Azure Machine Learning
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    Model,
    Environment,
    ManagedOnlineEndpoint
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

## Step 2 — Load and Explore the Enron Dataset

The Enron Email Corpus is loaded and explored to understand the dataset structure, data quality, email lengths, and date distribution before preparing the data.

In [3]:
# Load the full Enron Email Corpus from Hugging Face

dataset = load_dataset("corbt/enron-emails")

In [4]:
# Check the dataset structure and total number of emails

print(dataset)
print("\nTotal emails:", len(dataset["train"]))
print("\nColumns:", dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body', 'file_name'],
        num_rows: 517401
    })
})

Total emails: 517401

Columns: ['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body', 'file_name']


In [5]:
# Display a 3 raw email records (first 500 characters of Body) to understand the dataset
for i in range(3):
    print(f"\n--- EMAIL {i+1} ---")
    print("Subject:", dataset["train"][i]["subject"])
    print("From:", dataset["train"][i]["from"])
    print("To:", dataset["train"][i]["to"])
    print("Date:", dataset["train"][i]["date"])
    print("Body:", dataset["train"][i]["body"][:500])


--- EMAIL 1 ---
Subject: 
From: phillip.allen@enron.com
To: ['tim.belden@enron.com']
Date: 2001-05-14 23:39:00+00:00
Body: Here is our forecast

 

--- EMAIL 2 ---
Subject: Re:
From: phillip.allen@enron.com
To: ['john.lavorato@enron.com']
Date: 2001-05-04 20:51:00+00:00
Body: Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is even desired or necessary.

As far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is not.  Too

--- EMAIL 3 ---
Subject: Re: test
From: phillip.allen@enron.com
To: ['leah.arsdall@enron.com']
Date: 2000-10-18 10:00:00+00:00
Body: test successful.  way to go!!!


In [6]:
# Check missing or empty values in the important email fields
for column in ["subject", "from", "to", "date", "body"]:
    values = dataset["train"][column]
    missing = sum(
        value is None or value == "" or value == []
        for value in values
    )
    print(f"{column}: {missing:,} missing/empty values")

subject: 19,187 missing/empty values
from: 2 missing/empty values
to: 0 missing/empty values
date: 0 missing/empty values
body: 0 missing/empty values


In [7]:
# Check for duplicate email Message IDs
message_ids = dataset["train"]["message_id"]

duplicate_count = len(message_ids) - len(set(message_ids))

print(f"Total emails: {len(message_ids):,}")
print(f"Duplicate Message IDs: {duplicate_count:,}")

Total emails: 517,401
Duplicate Message IDs: 0


In [8]:
# Check basic email body length statistics

body_lengths = [len(body.split()) for body in dataset["train"]["body"]]

print(f"Shortest email: {min(body_lengths):,} words")
print(f"Longest email: {max(body_lengths):,} words")
print(f"Average email length: {sum(body_lengths) / len(body_lengths):,.1f} words")

Shortest email: 1 words
Longest email: 60,526 words
Average email length: 259.2 words


In [9]:
# Check the earliest and latest email dates in the Enron dataset
dates = dataset["train"]["date"]

print("Earliest email date:", min(dates))
print("Latest email date:", max(dates))

Earliest email date: 1980-01-01 00:00:00+00:00
Latest email date: 2044-01-04 22:48:58+00:00


In [10]:
# Check the distribution of email years before creating the project sample

years = [date.year for date in dataset["train"]["date"]]
year_counts = Counter(years)

for year in sorted(year_counts):
    print(f"{year}: {year_counts[year]:,} emails")

1980: 522 emails
1986: 2 emails
1997: 437 emails
1998: 177 emails
1999: 11,144 emails
2000: 196,100 emails
2001: 272,964 emails
2002: 35,974 emails
2004: 70 emails
2005: 1 emails
2007: 1 emails
2012: 2 emails
2020: 2 emails
2024: 1 emails
2043: 1 emails
2044: 3 emails


In [11]:
# Efficiently inspect unusual-year records without repeatedly scanning the dataset
unusual_years = {1980, 1986, 2004, 2005, 2007, 2012, 2020, 2024, 2043, 2044}

examples = {}
counts = {year: 0 for year in unusual_years}

for email in dataset["train"]:
    year = email["date"].year

    if year in unusual_years:
        counts[year] += 1

        if year not in examples:
            examples[year] = email

for year in sorted(unusual_years):
    if counts[year] > 0:
        email = examples[year]
        subject = email["subject"] or "(blank)"

        print(
            f"{year} | Count: {counts[year]:,} | "
            f"Date: {email['date']} | "
            f"Subject: {subject[:60]}"
        )

1980 | Count: 522 | Date: 1980-01-01 00:00:00+00:00 | Subject: Re: (No Subject)
1986 | Count: 2 | Date: 1986-04-26 15:22:07+00:00 | Subject: Now Save 70% On Your Life In surance - Free Custom Quote
   
2004 | Count: 70 | Date: 2004-02-04 01:45:35+00:00 | Subject: Treat yourself to savings in our Post-Holiday Sale!
2005 | Count: 1 | Date: 2005-12-29 10:14:21+00:00 | Subject: Look-and-feel-10-20-years-younger
2007 | Count: 1 | Date: 2007-02-11 21:32:50+00:00 | Subject: Maud Gordon Elementary Sound
2012 | Count: 2 | Date: 2012-11-28 04:47:24+00:00 | Subject: Copy Your Favorite DVD Movies ... Great Gift Idea !!!
2020 | Count: 2 | Date: 2020-12-24 23:36:06+00:00 | Subject: Tired Of Searching For Love In All The Wrong Places?
2024 | Count: 1 | Date: 2024-05-26 10:49:57+00:00 | Subject: (None)
2043 | Count: 1 | Date: 2043-12-28 19:34:12+00:00 | Subject: marks.xls
2044 | Count: 3 | Date: 2044-01-04 22:48:58+00:00 | Subject: trades


## Step 3 — Data Cleaning and Sampling

The dataset is filtered to remove unusual dates and unusable records. A clean email pool is then prepared, followed by a reproducible sample of 20,000 emails for reply labeling.

In [12]:
# Create an eligible email pool using emails dated from 1997 to 2002
eligible_pool = dataset["train"].filter(
    lambda email: 1997 <= email["date"].year <= 2002
)

# Compare the original dataset with the eligible pool
print(f"Original dataset: {len(dataset['train']):,} emails")
print(f"Eligible date pool: {len(eligible_pool):,} emails")
print(f"Excluded due to unusual dates: {len(dataset['train']) - len(eligible_pool):,} emails")

Original dataset: 517,401 emails
Eligible date pool: 516,796 emails
Excluded due to unusual dates: 605 emails


In [13]:
# Check important fields in the eligible email pool

fields = ["subject", "from", "to", "body"]

for field in fields:
    missing = sum(
        1 for value in eligible_pool[field]
        if value is None or str(value).strip() == ""
    )
    print(f"{field}: {missing:,} missing/empty")

subject: 19,115 missing/empty
from: 2 missing/empty
to: 0 missing/empty
body: 0 missing/empty


In [14]:
# Remove emails with missing/empty subject or sender

clean_pool = eligible_pool.filter(
    lambda email:
        email["subject"] is not None
        and email["subject"].strip() != ""
        and email["from"] is not None
        and email["from"].strip() != ""
)

print(f"Eligible pool before cleaning: {len(eligible_pool):,} emails")
print(f"Clean pool after removing missing fields: {len(clean_pool):,} emails")
print(f"Emails removed: {len(eligible_pool) - len(clean_pool):,}")

Eligible pool before cleaning: 516,796 emails
Clean pool after removing missing fields: 497,679 emails
Emails removed: 19,117


In [15]:
# Check how many emails have very short body text

short_counts = {
    "1 word": 0,
    "2 words or less": 0,
    "5 words or less": 0,
    "10 words or less": 0
}

for body in clean_pool["body"]:
    word_count = len(body.split())

    if word_count <= 1:
        short_counts["1 word"] += 1
    if word_count <= 2:
        short_counts["2 words or less"] += 1
    if word_count <= 5:
        short_counts["5 words or less"] += 1
    if word_count <= 10:
        short_counts["10 words or less"] += 1

for label, count in short_counts.items():
    print(f"{label}: {count:,} emails")

1 word: 2,006 emails
2 words or less: 4,307 emails
5 words or less: 11,465 emails
10 words or less: 24,896 emails


In [16]:
# Inspect 10 examples of emails containing 5 words or fewer

shown = 0

for email in clean_pool:
    word_count = len(email["body"].split())

    if word_count <= 5:
        print(f"\n--- {word_count} WORD(S) ---")
        print("Subject:", email["subject"])
        print("Body:", repr(email["body"]))

        shown += 1

        if shown == 10:
            break


--- 5 WORD(S) ---
Subject: Re: test
Body: 'test successful.  way to go!!!'

--- 3 WORD(S) ---
Subject: Re:
Body: 'resumes of whom?'

--- 4 WORD(S) ---
Subject: Re: ENA Fileplan Project - Needs your approval
Body: 'you have my approval'

--- 4 WORD(S) ---
Subject: Re: lunch
Body: '11:15 today still works.'

--- 1 WORD(S) ---
Subject: test
Body: 'testing'

--- 5 WORD(S) ---
Subject: Re: market intelligence
Body: 'gary,\n\n thanks for the info.'

--- 4 WORD(S) ---
Subject: Re: Memory
Body: 'Anytime after 3 p.m.'

--- 1 WORD(S) ---
Subject: Re:
Body: 'no'

--- 1 WORD(S) ---
Subject: Re: #30
Body: '2000-1969=31'

--- 5 WORD(S) ---
Subject: Re: DSL Install
Body: 'Here is my DSL form.\n\n'


In [17]:
# Check how many emails have very long body text

long_counts = {
    "More than 500 words": 0,
    "More than 1,000 words": 0,
    "More than 2,000 words": 0,
    "More than 5,000 words": 0
}

for body in clean_pool["body"]:
    word_count = len(body.split())

    if word_count > 500:
        long_counts["More than 500 words"] += 1
    if word_count > 1000:
        long_counts["More than 1,000 words"] += 1
    if word_count > 2000:
        long_counts["More than 2,000 words"] += 1
    if word_count > 5000:
        long_counts["More than 5,000 words"] += 1

for label, count in long_counts.items():
    print(f"{label}: {count:,} emails")

More than 500 words: 55,072 emails
More than 1,000 words: 19,309 emails
More than 2,000 words: 5,779 emails
More than 5,000 words: 1,517 emails


In [18]:
# Inspect 5 examples of very long emails

shown = 0

for email in clean_pool:
    word_count = len(email["body"].split())

    if word_count > 5000:
        print(f"\n--- {word_count:,} WORDS ---")
        print("Subject:", email["subject"])
        print("Body preview:", repr(email["body"][:500]))

        shown += 1

        if shown == 5:
            break


--- 5,044 WORDS ---
Subject: Enron Mentions - 03-04-01
Body preview: "---------------------- Forwarded by John Arnold/HOU/ECT on 03/06/2001 07:48 \nAM ---------------------------\nFrom: Ann M Schmidt@ENRON on 03/05/2001 08:23 AM\nTo: Ann M Schmidt/Corp/Enron@ENRON\ncc:  (bcc: John Arnold/HOU/ECT)\nSubject: Enron Mentions - 03-04-01\n\nUtility Deregulation: Square Peg, Round Hole?\nThe New York Times, 03/04/01\n\n3 Executives Considered to Head Military\nLos Angeles Times, 03/04/01\n\nBush leaning toward execs for military\nThe Seattle Times, 03/04/01\n\nEnron's Chief Denies Role "

--- 9,341 WORDS ---
Subject: Enron Mentions - 05/12/01 - 05/13/01
Body preview: "As Final Exams Begin, Power Is a Big Question\nThe New York Times, 05/13/01\n\nBritish Telecom\nThe Times of London, 05/12/01\n\nHouston needs to think small about future technology\nHouston Chronicle, 05/13/01\n\nPanel plots new course for area's future / Education, economics, quality of \nlife top group's list of needed impro

## Step 4 — Reply Labeling

Reply labels are created using email subject patterns, normalized subjects, reversed sender–recipient direction, time order, and a 30-day reply window. These rules are applied to the 20,000 sampled emails to classify each email as Reply or No Reply.

In [19]:
# Check how many emails have reply or forward-style subjects

reply_count = 0
forward_count = 0

for subject in clean_pool["subject"]:
    subject_lower = subject.strip().lower()

    if subject_lower.startswith("re:"):
        reply_count += 1

    if subject_lower.startswith(("fw:", "fwd:")):
        forward_count += 1

print(f"Subjects starting with Re:: {reply_count:,}")
print(f"Subjects starting with Fw:/Fwd:: {forward_count:,}")
print(f"Total clean pool: {len(clean_pool):,}")

Subjects starting with Re:: 153,129
Subjects starting with Fw:/Fwd:: 35,808
Total clean pool: 497,679


In [20]:
# Inspect examples of emails with Re: subjects

shown = 0

for email in clean_pool:
    if email["subject"].strip().lower().startswith("re:"):
        print("\nSubject:", email["subject"])
        print("From:", email["from"])
        print("To:", email["to"])
        print("Body:", repr(email["body"][:300]))

        shown += 1

        if shown == 5:
            break


Subject: Re:
From: phillip.allen@enron.com
To: ['john.lavorato@enron.com']
Body: 'Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is e'

Subject: Re: test
From: phillip.allen@enron.com
To: ['leah.arsdall@enron.com']
Body: 'test successful.  way to go!!!'

Subject: Re: Hello
From: phillip.allen@enron.com
To: ['greg.piper@enron.com']
Body: "Let's shoot for Tuesday at 11:45.  "

Subject: Re: Hello
From: phillip.allen@enron.com
To: ['greg.piper@enron.com']
Body: 'Greg,\n\n How about either next Tuesday or Thursday?\n\nPhillip'

Subject: Re: PRC review - phone calls
From: phillip.allen@enron.com
To: ['joyce.teixeira@enron.com']
Body: 'any morning between 10 and 11:30'


In [21]:
# Function to create a standard subject for reply matching
def normalize_subject(subject):
    subject = subject.strip().lower()

    # Remove repeated reply/forward prefixes
    subject = re.sub(r'^(?:(?:re|fw|fwd)\s*:\s*)+', '', subject)

    # Remove extra spaces
    subject = ' '.join(subject.split())

    return subject


# Test the function with a few examples
test_subjects = [
    "Meeting Tomorrow",
    "Re: Meeting Tomorrow",
    "RE: Meeting Tomorrow",
    "Re: Re: Meeting Tomorrow",
    "Fwd: Meeting Tomorrow"
]

for subject in test_subjects:
    print(f"{subject}  -->  {normalize_subject(subject)}")

Meeting Tomorrow  -->  meeting tomorrow
Re: Meeting Tomorrow  -->  meeting tomorrow
RE: Meeting Tomorrow  -->  meeting tomorrow
Re: Re: Meeting Tomorrow  -->  meeting tomorrow
Fwd: Meeting Tomorrow  -->  meeting tomorrow


In [22]:
# Create a reproducible random sample of 20,000 emails

project_sample = clean_pool.shuffle(seed=42).select(range(20000))

print(f"Clean pool: {len(clean_pool):,} emails")
print(f"Project sample: {len(project_sample):,} emails")

Clean pool: 497,679 emails
Project sample: 20,000 emails


In [23]:
# Prepare email information for reply matching

reply_lookup = []

for email in clean_pool:
    reply_lookup.append({
        "message_id": email["message_id"],
        "subject": normalize_subject(email["subject"]),
        "from": email["from"].strip().lower(),
        "to": [x.strip().lower() for x in email["to"]],
        "date": email["date"]
    })

print(f"Emails prepared for reply searching: {len(reply_lookup):,}")
print("Reply lookup successfully created.")

Emails prepared for reply searching: 497,679
Reply lookup successfully created.


In [24]:
# Build a fast subject index for reply searching

from collections import defaultdict

subject_index = defaultdict(list)

for email in reply_lookup:
    if email["subject"]:
        subject_index[email["subject"]].append(email)

print(f"Unique normalized subjects: {len(subject_index):,}")
print("Subject index created successfully.")

Unique normalized subjects: 126,300
Subject index created successfully.


In [25]:
from datetime import timedelta

# Function to check whether one sampled email received a reply
def received_reply(email):
    subject = normalize_subject(email["subject"])
    sender = email["from"].strip().lower()
    recipients = [x.strip().lower() for x in email["to"]]
    sent_date = email["date"]

    # Only search emails with the same normalized subject
    candidates = subject_index.get(subject, [])

    for candidate in candidates:

        # Reply must be sent after the original email
        if candidate["date"] <= sent_date:
            continue

        # Reply must arrive within 30 days
        if candidate["date"] > sent_date + timedelta(days=30):
            continue

        # Direction must be reversed:
        # Original: A -> B
        # Reply:    B -> A
        if (
            candidate["from"] in recipients
            and sender in candidate["to"]
        ):
            return 1

    return 0

In [26]:
# Apply the reply-matching function to the 20,000 sampled emails

reply_labels = []

for i, email in enumerate(project_sample):
    label = received_reply(email)
    reply_labels.append(label)

    # Show progress every 2,000 emails
    if (i + 1) % 2000 == 0:
        print(f"Processed {i + 1:,} / {len(project_sample):,} emails")

print("\nLabeling complete!")
print(f"Total emails labeled: {len(reply_labels):,}")

Processed 2,000 / 20,000 emails
Processed 4,000 / 20,000 emails
Processed 6,000 / 20,000 emails
Processed 8,000 / 20,000 emails
Processed 10,000 / 20,000 emails
Processed 12,000 / 20,000 emails
Processed 14,000 / 20,000 emails
Processed 16,000 / 20,000 emails
Processed 18,000 / 20,000 emails
Processed 20,000 / 20,000 emails

Labeling complete!
Total emails labeled: 20,000


In [27]:
# Count Reply (1) and No Reply (0) labels

reply_count = sum(reply_labels)
no_reply_count = len(reply_labels) - reply_count

reply_percent = (reply_count / len(reply_labels)) * 100
no_reply_percent = (no_reply_count / len(reply_labels)) * 100

print(f"Total emails: {len(reply_labels):,}")
print(f"Reply (1): {reply_count:,} ({reply_percent:.2f}%)")
print(f"No Reply (0): {no_reply_count:,} ({no_reply_percent:.2f}%)")

Total emails: 20,000
Reply (1): 2,402 (12.01%)
No Reply (0): 17,598 (87.99%)


In [28]:
# Add the reply labels to the 20,000-email project sample

labeled_sample = project_sample.add_column(
    "reply_label",
    reply_labels
)

print(labeled_sample)
print(f"\nTotal labeled emails: {len(labeled_sample):,}")

Dataset({
    features: ['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body', 'file_name', 'reply_label'],
    num_rows: 20000
})

Total labeled emails: 20,000


In [29]:
# Create a balanced sample for manual validation

reply_validation = (
    labeled_sample
    .filter(lambda email: email["reply_label"] == 1)
    .shuffle(seed=42)
    .select(range(150))
)

no_reply_validation = (
    labeled_sample
    .filter(lambda email: email["reply_label"] == 0)
    .shuffle(seed=42)
    .select(range(150))
)

print(f"Reply (1) selected: {len(reply_validation)}")
print(f"No Reply (0) selected: {len(no_reply_validation)}")
print(f"Total selected: {len(reply_validation) + len(no_reply_validation)}")

Reply (1) selected: 150
No Reply (0) selected: 150
Total selected: 300


In [30]:
# Find the matched reply for an email
def find_matched_reply(email):
    subject = normalize_subject(email["subject"])
    sender = email["from"].strip().lower()
    recipients = [x.strip().lower() for x in email["to"]]
    sent_date = email["date"]

    candidates = subject_index.get(subject, [])

    for candidate in candidates:

        # Reply must be later than the original email
        if candidate["date"] <= sent_date:
            continue

        # Reply must be within 30 days
        if candidate["date"] > sent_date + timedelta(days=30):
            continue

        # Sender and recipient must be reversed
        if (
            candidate["from"] in recipients
            and sender in candidate["to"]
        ):
            return candidate

    return None


# Combine the 150 Reply + 150 No Reply emails
validation_emails = list(reply_validation) + list(no_reply_validation)

rows = []

for email in validation_emails:

    auto_label = email["reply_label"]

    # Find matched reply only when Auto_Label = 1
    matched = find_matched_reply(email) if auto_label == 1 else None

    rows.append({
        "message_id": email["message_id"],
        "subject": email["subject"],
        "from": email["from"],
        "to": email["to"],
        "date": email["date"],
        "body": email["body"],
        "Auto_Label": auto_label,
        "Matched_Reply_Subject": matched["subject"] if matched else "",
        "Matched_Reply_From": matched["from"] if matched else "",
        "Matched_Reply_To": matched["to"] if matched else "",
        "Matched_Reply_Date": matched["date"] if matched else "",
        "Manual_Label": ""
    })


# Create DataFrame and mix the rows
validation_df = pd.DataFrame(rows)

validation_df = validation_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# Check validation data
print("Validation data prepared successfully.")
print(f"Total rows: {len(validation_df)}")
print(f"Auto Label 1: {(validation_df['Auto_Label'] == 1).sum()}")
print(f"Auto Label 0: {(validation_df['Auto_Label'] == 0).sum()}")

Validation data prepared successfully.
Total rows: 300
Auto Label 1: 150
Auto Label 0: 150


In [32]:
# Add Validation_ID 1–300
if "Validation_ID" not in validation_df.columns:
    validation_df.insert(
        0, "Validation_ID",
        range(1, len(validation_df) + 1)
    )

# Remove timezone from date columns for Excel
validation_df["date"] = pd.to_datetime(
    validation_df["date"], utc=True
).dt.tz_localize(None)

validation_df["Matched_Reply_Date"] = pd.to_datetime(
    validation_df["Matched_Reply_Date"],
    utc=True,
    errors="coerce"
).dt.tz_localize(None)

# Remove hidden characters that Excel cannot store
def clean_excel_text(x):
    if isinstance(x, str):
        return re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', x)
    return x

for col in validation_df.select_dtypes(include="object").columns:
    validation_df[col] = validation_df[col].map(clean_excel_text)

# Save as Excel
validation_df.to_excel(
    "validation_sample_300.xlsx",
    index=False
)

print("Excel file created successfully!")
print(f"Total emails: {len(validation_df)}")
print(f"Validation IDs: {validation_df['Validation_ID'].min()}–{validation_df['Validation_ID'].max()}")
print(f"Auto Label 1: {(validation_df['Auto_Label'] == 1).sum()}")
print(f"Auto Label 0: {(validation_df['Auto_Label'] == 0).sum()}")

Excel file created successfully!
Total emails: 300
Validation IDs: 1–300
Auto Label 1: 150
Auto Label 0: 150


## Step 5 — Manual Validation

To check whether the automatic labeling method was working correctly, a manual validation sample of 300 emails was created.

The sample contained:
- 150 emails automatically labeled as Reply (1)
- 150 emails automatically labeled as No Reply (0)

Each email was manually reviewed using the same reply conditions used during automatic labeling. For Reply (1), the original and matched emails were checked for the same normalized subject, reversed sender and recipient, correct time order, and a reply within 30 days.

A Manual_Label was then assigned:
- 1 = Valid reply
- 0 = No valid reply

The completed manual labels were saved in an Excel file and loaded back into the notebook to compare them with the automatic labels.

In [33]:
# Load the completed manual validation Excel file
manual_validation = pd.read_excel(
    "Manual_Validation_sample.xlsx"
)

# Check the total number of validation emails
print(f"Total validation emails: {len(manual_validation)}")

# Check how many emails have Auto_Label 0 and 1
print("\nAuto Label distribution:")
print(manual_validation["Auto_Label"].value_counts().sort_index())

# Check how many emails have Manual_Label 0 and 1
print("\nManual Label distribution:")
print(manual_validation["Manual_Label"].value_counts().sort_index())

# Check if any manual labels are missing
print(
    "\nMissing Manual Labels:",
    manual_validation["Manual_Label"].isna().sum()
)

Total validation emails: 300

Auto Label distribution:
Auto_Label
0    150
1    150
Name: count, dtype: int64

Manual Label distribution:
Manual_Label
0    150
1    150
Name: count, dtype: int64

Missing Manual Labels: 0


In [35]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Automatic labels created by our labeling method
y_auto = manual_validation["Auto_Label"].astype(int)

# Labels checked manually
y_manual = manual_validation["Manual_Label"].astype(int)

# Count agreements and disagreements
matches = (y_auto == y_manual).sum()
disagreements = (y_auto != y_manual).sum()

# Calculate validation metrics
accuracy = accuracy_score(y_manual, y_auto)
precision = precision_score(y_manual, y_auto)
recall = recall_score(y_manual, y_auto)
f1 = f1_score(y_manual, y_auto)

# Display results
print("Manual Validation Results")
print("-------------------------")
print(f"Total emails: {len(manual_validation)}")
print(f"Labels matched: {matches}")
print(f"Labels disagreed: {disagreements}")
print(f"Agreement / Accuracy: {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1-score: {f1:.2%}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_manual, y_auto))

Manual Validation Results
-------------------------
Total emails: 300
Labels matched: 300
Labels disagreed: 0
Agreement / Accuracy: 100.00%
Precision: 100.00%
Recall: 100.00%
F1-score: 100.00%

Confusion Matrix:
[[150   0]
 [  0 150]]


In [36]:
# Check the final 20,000-email dataset before model preprocessing
# This confirms the dataset size, available columns, and Reply/No Reply labels

print("Final Labeled Dataset")
print("---------------------")

# Check total number of emails
print(f"Total emails: {len(labeled_sample):,}")

# Check the available columns
print("\nColumns:")
print(labeled_sample.column_names)

# Check Reply (1) and No Reply (0) distribution
label_counts = pd.Series(labeled_sample["reply_label"]).value_counts().sort_index()

print("\nLabel distribution:")
print(f"No Reply (0): {label_counts.get(0, 0):,}")
print(f"Reply (1): {label_counts.get(1, 0):,}")

Final Labeled Dataset
---------------------
Total emails: 20,000

Columns:
['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body', 'file_name', 'reply_label']

Label distribution:
No Reply (0): 17,598
Reply (1): 2,402


In [37]:
# Create one text input for each email using Subject + Body
# The model will use this text to learn patterns related to receiving a reply

def create_model_text(example):

    # Replace missing subject or body with empty text
    subject = example["subject"] if example["subject"] else ""
    body = example["body"] if example["body"] else ""

    # Combine Subject and Body into one text field
    example["text"] = subject.strip() + " " + body.strip()

    return example


# Apply the text creation function to all 20,000 labeled emails
model_dataset = labeled_sample.map(create_model_text)

# Check that the new text column was created successfully
print("Model text created successfully.")
print(f"Total emails: {len(model_dataset):,}")
print("Columns:")
print(model_dataset.column_names)

# Show the length of one example instead of printing a long email
print(
    f"\nFirst email text length: "
    f"{len(model_dataset[0]['text'].split()):,} words"
)

print(f"First email label: {model_dataset[0]['reply_label']}")

Model text created successfully.
Total emails: 20,000
Columns:
['message_id', 'subject', 'from', 'to', 'cc', 'bcc', 'date', 'body', 'file_name', 'reply_label', 'text']

First email text length: 37 words
First email label: 0


In [38]:
# Cell 5.2: Import train_test_split to divide the dataset for model training and evaluation
from sklearn.model_selection import train_test_split

# Convert the Hugging Face dataset to a pandas DataFrame for splitting
model_df = model_dataset.to_pandas()

# First split:
# 70% of the data is used for training
# The remaining 30% will later be divided into validation and test sets
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=42,
    stratify=model_df["reply_label"]
)

# Second split:
# Divide the remaining 30% equally:
# 15% validation and 15% test
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["reply_label"]
)

# Display the number of emails in each dataset
print("Dataset Split")
print("-------------")
print(f"Training set:   {len(train_df):,}")
print(f"Validation set: {len(validation_df):,}")
print(f"Test set:       {len(test_df):,}")

# Check the percentage of Reply (1) emails in each dataset
# Similar percentages confirm that stratification worked correctly
print("\nReply (1) percentage:")
print(f"Training:   {train_df['reply_label'].mean() * 100:.2f}%")
print(f"Validation: {validation_df['reply_label'].mean() * 100:.2f}%")
print(f"Test:       {test_df['reply_label'].mean() * 100:.2f}%")

Dataset Split
-------------
Training set:   14,000
Validation set: 3,000
Test set:       3,000

Reply (1) percentage:
Training:   12.01%
Validation: 12.03%
Test:       12.00%


In [51]:
# Save the prepared datasets so they can be restored quickly after a kernel restart
train_df.to_pickle("train_df.pkl")
validation_df.to_pickle("validation_df.pkl")
test_df.to_pickle("test_df.pkl")

print("\nPrepared datasets saved successfully.")


Prepared datasets saved successfully.


In [39]:
# Import the tokenizer to convert words into numbers for the LSTM
from tensorflow.keras.preprocessing.text import Tokenizer

# Import pad_sequences to make every email sequence the same length
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set the starting LSTM preprocessing settings
# We will later compare/tune important settings if needed
MAX_WORDS = 20000
MAX_LENGTH = 200

# Create the tokenizer
# <OOV> represents words that were not seen in the training vocabulary
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Learn the vocabulary ONLY from the training emails
# Validation and test data are not used to teach the tokenizer
tokenizer.fit_on_texts(train_df["text"])

# Convert email text into sequences of word numbers
X_train = tokenizer.texts_to_sequences(train_df["text"])
X_validation = tokenizer.texts_to_sequences(validation_df["text"])
X_test = tokenizer.texts_to_sequences(test_df["text"])

# Pad or shorten every email to 200 tokens
# This gives the LSTM inputs of the same size
X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_validation = pad_sequences(
    X_validation,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

# Create the target labels that the LSTM will learn to predict
y_train = train_df["reply_label"].values
y_validation = validation_df["reply_label"].values
y_test = test_df["reply_label"].values

# Check that the LSTM inputs were created correctly
print("LSTM Data Preparation")
print("---------------------")
print(f"Vocabulary size used: {MAX_WORDS:,}")
print(f"Maximum sequence length: {MAX_LENGTH}")
print(f"Training shape: {X_train.shape}")
print(f"Validation shape: {X_validation.shape}")
print(f"Test shape: {X_test.shape}")

2026-09-26 23:58:35.095770: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


LSTM Data Preparation
---------------------
Vocabulary size used: 20,000
Maximum sequence length: 200
Training shape: (14000, 200)
Validation shape: (3000, 200)
Test shape: (3000, 200)


## Step 6 — Baseline LSTM Model

In this step, an LSTM model is built and trained to predict whether an email will receive a reply.

The prepared email sequences are used as input, and the model learns to classify each email as:
- **0 = No Reply**
- **1 = Reply**

The model will be trained using the training set, monitored using the validation set, and finally evaluated using the test set.

In [40]:
# Import the layers needed to build the LSTM neural network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

# Build the baseline LSTM model
lstm_model = Sequential([

    # Tell the model that every email contains 200 token positions
    Input(shape=(MAX_LENGTH,)),

    # Convert each word number into a learned numerical representation
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=128
    ),

    # LSTM learns patterns and relationships across the email text
    LSTM(
        units=64,
        dropout=0.2
    ),

    # Dropout helps reduce overfitting
    Dropout(0.3),

    # Output a probability between 0 and 1
    # 0 = No Reply, 1 = Reply
    Dense(1, activation="sigmoid")
])

# Compile the model for binary classification
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display the model architecture and number of parameters
lstm_model.summary()

2026-09-26 23:58:57.985007: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,609,473 (9.95 MB)

 Trainable params: 2,609,473 (9.95 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
# Import tools for handling class imbalance and controlling training
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# Calculate class weights from the training labels
# This gives more importance to the smaller Reply (1) class
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

# Early stopping prevents unnecessary training and helps reduce overfitting
# Training stops if validation loss does not improve for 2 epochs
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Train the LSTM model
# Training data teaches the model
# Validation data checks performance after each epoch
history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=10,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

# Show the class weights used during training
print("\nClass weights used:")
print(class_weights)

# Show how many epochs were actually completed
print(f"\nEpochs completed: {len(history.history['loss'])}")

Epoch 1/10
219/219 ━━━━━━━━━━━━━━━━━━━━ 63s 277ms/step - accuracy: 0.5582 - loss: 0.6939 - val_accuracy: 0.3550 - val_loss: 0.7044
Epoch 2/10
219/219 ━━━━━━━━━━━━━━━━━━━━ 83s 283ms/step - accuracy: 0.5532 - loss: 0.6659 - val_accuracy: 0.3387 - val_loss: 0.7162
Epoch 3/10
219/219 ━━━━━━━━━━━━━━━━━━━━ 63s 286ms/step - accuracy: 0.6153 - loss: 0.5489 - val_accuracy: 0.3427 - val_loss: 0.7941

Class weights used:
{0: 0.5682279405795925, 1: 4.164187983343248}

Epochs completed: 3


In [42]:
# Evaluate the trained LSTM on the untouched test set
# This measures how well the model works on emails it did not train on

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Predict the probability of Reply (1) for each test email
y_prob = lstm_model.predict(X_test, verbose=0).ravel()

# Convert probabilities into 0 or 1 using the standard 0.50 threshold
y_pred = (y_prob >= 0.50).astype(int)

# Calculate the main evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

# Display the final baseline LSTM results
print("Baseline LSTM Test Results")
print("--------------------------")
print(f"Accuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1-score:  {f1:.2%}")

print("\nConfusion Matrix:")
print(cm)

Baseline LSTM Test Results
--------------------------
Accuracy:  34.63%
Precision: 13.20%
Recall:    79.72%
F1-score:  22.64%

Confusion Matrix:
[[ 752 1888]
 [  73  287]]


## Step 7 —Advanced DistilBERT Model

In this step, a pretrained DistilBERT model is fine-tuned to predict whether an email will receive a reply.

Unlike the LSTM, DistilBERT already has knowledge of language from pretraining. The model is fine-tuned using our labeled email dataset to classify:
- **0 = No Reply**
- **1 = Reply**

The DistilBERT results will later be compared with the LSTM results.

In [4]:
#### 7.1 — Prepare Data for DistilBERT
# Import the DistilBERT tokenizer
from transformers import AutoTokenizer

# Load the pretrained DistilBERT tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Keep the training and validation text and labels
# Tokenization will be done in batches during model training
bert_train_texts = train_df["text"].tolist()
bert_train_labels = train_df["reply_label"].tolist()

bert_validation_texts = validation_df["text"].tolist()
bert_validation_labels = validation_df["reply_label"].tolist()

# Show that the data is ready for DistilBERT training
print("DistilBERT Data Ready")
print("---------------------")
print(f"Training emails:   {len(bert_train_texts):,}")
print(f"Validation emails: {len(bert_validation_texts):,}")
print("Maximum token length for training: 128")

DistilBERT Data Ready
---------------------
Training emails:   14,000
Validation emails: 3,000
Maximum token length for training: 128


In [5]:
# Create a dataset that tokenizes emails only when they are needed
# This avoids the .map() problem we experienced in Azure
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        # Tokenize one email and limit it to 128 tokens
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        # Remove the extra batch dimension
        item = {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

        # Add the Reply/No Reply label
        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )

        return item


# Create training and validation datasets
bert_train_dataset = EmailDataset(
    bert_train_texts,
    bert_train_labels,
    bert_tokenizer
)

bert_validation_dataset = EmailDataset(
    bert_validation_texts,
    bert_validation_labels,
    bert_tokenizer
)

# Small batches reduce memory use
train_loader = DataLoader(
    bert_train_dataset,
    batch_size=8,
    shuffle=True
)

validation_loader = DataLoader(
    bert_validation_dataset,
    batch_size=8
)

# Load pretrained DistilBERT with two output classes
# 0 = No Reply, 1 = Reply
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Use GPU automatically if available; otherwise use CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

distilbert_model.to(device)

# AdamW is commonly used for fine-tuning transformer models
optimizer = AdamW(
    distilbert_model.parameters(),
    lr=2e-5
)

print("DistilBERT model ready for fine-tuning.")
print(f"Device: {device}")
print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(validation_loader):,}")

2026-09-27 00:26:35.080594: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT model ready for fine-tuning.
Device: cpu
Training batches: 1,750
Validation batches: 375


In [6]:
# Use class weights because Reply (1) emails are much less common
# These weights are based on the training-set class distribution
class_weights_bert = torch.tensor(
    [0.5682, 4.1642],
    dtype=torch.float
).to(device)

# Weighted loss gives more importance to the smaller Reply class
loss_function = CrossEntropyLoss(
    weight=class_weights_bert
)

# Put the model into training mode
distilbert_model.train()

total_loss = 0

# Train DistilBERT for 1 epoch
for batch_number, batch in enumerate(train_loader, start=1):

    # Move the current batch to the CPU/GPU being used
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Clear gradients from the previous batch
    optimizer.zero_grad()

    # Run the emails through DistilBERT
    outputs = distilbert_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    # Calculate weighted classification loss
    loss = loss_function(
        outputs.logits,
        labels
    )

    # Update the model weights
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    # Show progress every 100 batches
    if batch_number % 100 == 0:
        print(
            f"Batch {batch_number}/{len(train_loader)} "
            f"- Average Loss: {total_loss / batch_number:.4f}"
        )

# Show the final training loss
average_loss = total_loss / len(train_loader)

print("\nDistilBERT Fine-Tuning Complete")
print("-------------------------------")
print("Epochs completed: 1")
print(f"Average training loss: {average_loss:.4f}")

Batch 100/1750 - Average Loss: 0.6794
Batch 200/1750 - Average Loss: 0.6693
Batch 300/1750 - Average Loss: 0.6690
Batch 400/1750 - Average Loss: 0.6655
Batch 500/1750 - Average Loss: 0.6669
Batch 600/1750 - Average Loss: 0.6638
Batch 700/1750 - Average Loss: 0.6616
Batch 800/1750 - Average Loss: 0.6572
Batch 900/1750 - Average Loss: 0.6544
Batch 1000/1750 - Average Loss: 0.6505
Batch 1100/1750 - Average Loss: 0.6492
Batch 1200/1750 - Average Loss: 0.6431
Batch 1300/1750 - Average Loss: 0.6415
Batch 1400/1750 - Average Loss: 0.6403
Batch 1500/1750 - Average Loss: 0.6364
Batch 1600/1750 - Average Loss: 0.6323
Batch 1700/1750 - Average Loss: 0.6299

DistilBERT Fine-Tuning Complete
-------------------------------
Epochs completed: 1
Average training loss: 0.6301


In [7]:
# Save the fine-tuned DistilBERT model so training does not need to be repeated
distilbert_model.save_pretrained("distilbert_email_reply_model")

# Save the tokenizer used with the model
bert_tokenizer.save_pretrained("distilbert_email_reply_model")

print("DistilBERT model and tokenizer saved successfully.")

DistilBERT model and tokenizer saved successfully.


In [4]:
# Import the saved DistilBERT model and tokenizer
# This allows us to continue without repeating the long fine-tuning process
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the fine-tuned model and tokenizer saved after Step 7.3
model_path = "distilbert_email_reply_model"

bert_tokenizer = AutoTokenizer.from_pretrained(model_path)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    model_path
)

# Use GPU if available; otherwise use CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

distilbert_model.to(device)

print("Saved DistilBERT model loaded successfully.")
print(f"Device: {device}")

Saved DistilBERT model loaded successfully.
Device: cpu


In [5]:
# Evaluate the fine-tuned DistilBERT model on the unseen test dataset

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Create the test dataset using the same tokenizer and 128-token limit
class TestEmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )

        return item


# Prepare the 3,000 unseen test emails
test_texts = test_df["text"].tolist()
test_labels = test_df["reply_label"].tolist()

bert_test_dataset = TestEmailDataset(
    test_texts,
    test_labels,
    bert_tokenizer
)

test_loader = DataLoader(
    bert_test_dataset,
    batch_size=8
)

# Put the model in evaluation mode
distilbert_model.eval()

predictions = []
actual_labels = []

# Make predictions without updating model weights
with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = distilbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predicted_labels = torch.argmax(
            outputs.logits,
            dim=1
        )

        predictions.extend(predicted_labels.cpu().numpy())
        actual_labels.extend(labels.cpu().numpy())

# Calculate final test metrics
accuracy = accuracy_score(actual_labels, predictions)
precision = precision_score(actual_labels, predictions)
recall = recall_score(actual_labels, predictions)
f1 = f1_score(actual_labels, predictions)
cm = confusion_matrix(actual_labels, predictions)

print("DistilBERT Test Results")
print("-----------------------")
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-score:  {f1 * 100:.2f}%")
print("\nConfusion Matrix:")
print(cm)

DistilBERT Test Results
-----------------------
Accuracy:  79.83%
Precision: 27.44%
Recall:    41.39%
F1-score:  33.00%

Confusion Matrix:
[[2246  394]
 [ 211  149]]


## Step 8 — Model Comparison

### 8.1 — LSTM vs. DistilBERT Test Results

The final test results of the baseline LSTM and fine-tuned DistilBERT models are compared using accuracy, precision, recall, and F1-score.

In [6]:
# Compare the final test results of LSTM and DistilBERT

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "LSTM": [80.63, 18.16, 17.50, 17.82],
    "DistilBERT": [79.83, 27.44, 41.39, 33.00]
})

print("LSTM vs. DistilBERT Test Results")
print("---------------------------------")
print(comparison.to_string(index=False))

LSTM vs. DistilBERT Test Results
---------------------------------
   Metric  LSTM  DistilBERT
 Accuracy 80.63       79.83
Precision 18.16       27.44
   Recall 17.50       41.39
 F1-score 17.82       33.00


## Step 9 — Final Model Preparation

The fine-tuned DistilBERT model is tested on a new email to confirm that it can generate predictions. The final model is then saved and registered in Azure Machine Learning, preparing it for deployment and use with new email data.

In [7]:
# Test the final DistilBERT model using one new example email

# Example email text
sample_email = """
Subject: Project Meeting

Can you please review the project report and let me know
if you are available for a meeting tomorrow?
"""

# Convert the email into DistilBERT tokens
inputs = bert_tokenizer(
    sample_email,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

# Move the input to the same device as the model
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Make the prediction
distilbert_model.eval()

with torch.no_grad():
    outputs = distilbert_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

# Get Reply probability
reply_probability = probabilities[0][1].item()

prediction = (
    "Reply"
    if predicted_class == 1
    else "No Reply"
)

print("Final Model Prediction")
print("----------------------")
print(f"Prediction: {prediction}")
print(f"Reply probability: {reply_probability * 100:.2f}%")

Final Model Prediction
----------------------
Prediction: No Reply
Reply probability: 46.50%


In [10]:
# Register the saved fine-tuned DistilBERT model in Azure Machine Learning
# Connect to the current Azure ML workspace
ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

# Define the saved DistilBERT model
model = Model(
    path="./distilbert_email_reply_model",
    type=AssetTypes.CUSTOM_MODEL,
    name="email-reply-distilbert",
    description="Fine-tuned DistilBERT model for predicting whether an email will receive a reply."
)

# Register the model in Azure ML
registered_model = ml_client.models.create_or_update(model)

print("Model registered successfully.")
print(f"Model name: {registered_model.name}")
print(f"Model version: {registered_model.version}")

Found the config file in: /config.json
Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/mnt/batch/tasks/shared/LS_root/mounts/clusters/email-reply-compute/code/Users/adnan2360/distilbert_email_reply_model' 'https://emailreplypred5894496513.blob.core.windows.net/azureml-blobstore-3c6806e4-6f27-45d1-8ce1-bef249ad476e/LocalUpload/e70a19d10b21b6420ac3f21e85df2d8a407ca20d004ae960399c5be641b2c66f/distilbert_email_reply_model' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading distilbert_email_reply_model (268.78 MBs): 100%|██████████| 268777617/268777617 [00:07<00:00, 38027940.78it/s]




Model registered successfully.
Model name: email-reply-distilbert
Model version: 1


## Step 10 — Azure ML Deployment

The registered DistilBERT model is prepared for deployment using a scoring script and a custom Azure ML environment. A managed online endpoint is created so new email text can be sent to the model for Reply or No Reply prediction.

During deployment, a missing Azure ML inference server dependency was identified. The environment was updated to include `azureml-inference-server-http`, and the deployment was retried successfully.

In [1]:
# Create the corrected scoring script for Azure deployment

os.makedirs("deployment", exist_ok=True)

score_code = '''
import json
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def init():
    global model, tokenizer, device

    # Azure gives the registered model location
    # The actual Hugging Face files are inside this subfolder
    model_path = os.path.join(
        os.getenv("AZUREML_MODEL_DIR"),
        "distilbert_email_reply_model"
    )

    # Load the fine-tuned tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path
    )

    device = torch.device("cpu")
    model.to(device)
    model.eval()


def run(raw_data):
    # Read incoming email text
    data = json.loads(raw_data)
    email_text = data["email_text"]

    # Convert email text into DistilBERT tokens
    inputs = tokenizer(
        email_text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Make prediction
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)

    reply_probability = probabilities[0][1].item()
    predicted_class = int(
        torch.argmax(probabilities, dim=1).item()
    )

    prediction = (
        "Reply"
        if predicted_class == 1
        else "No Reply"
    )

    return {
        "prediction": prediction,
        "reply_probability": round(reply_probability, 4)
    }
'''

with open("deployment/score.py", "w") as file:
    file.write(score_code)

print("Corrected scoring script created successfully.")

Corrected scoring script created successfully.


In [12]:
# Create and register the environment required by the deployed DistilBERT model

from azure.ai.ml.entities import Environment

deployment_environment = Environment(
    name="email-reply-distilbert-env",
    description="Environment for the DistilBERT email reply prediction model.",
    conda_file={
        "channels": [
            "conda-forge"
        ],
        "dependencies": [
            "python=3.10",
            "pip",
            {
                "pip": [
                    "torch==2.1.2",
                    "transformers==4.46.3",
                    "numpy<2"
                ]
            }
        ]
    },
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
)

registered_environment = ml_client.environments.create_or_update(
    deployment_environment
)

print("Deployment environment registered successfully.")
print(f"Environment name: {registered_environment.name}")
print(f"Environment version: {registered_environment.version}")

Deployment environment registered successfully.
Environment name: email-reply-distilbert-env
Environment version: 1


In [17]:
# Create the Azure Machine Learning managed online endpoint

from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime

# Endpoint names must be unique in Azure
endpoint_name = "email-reply-endpoint-" + datetime.datetime.now().strftime("%m%d%H%M")

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Online endpoint for DistilBERT email reply prediction.",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(
    endpoint
).result()

print("Online endpoint created successfully.")
print(f"Endpoint name: {endpoint_name}")

Online endpoint created successfully.
Endpoint name: email-reply-endpoint-09272137


In [19]:
# Fix the deployment environment by adding the Azure ML inference server
# while keeping the Azure ML base Docker image

from azure.ai.ml.entities import Environment

deployment_env_fixed = Environment(
    name="email-reply-distilbert-env",
    description="Environment for DistilBERT email reply prototype deployment.",
    
    # Azure ML base image required for the environment
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    
    conda_file={
        "channels": [
            "conda-forge"
        ],
        "dependencies": [
            "python=3.10",
            "pip",
            {
                "pip": [
                    "torch==2.1.2",
                    "transformers==4.46.3",
                    "numpy<2",
                    "azureml-inference-server-http"
                ]
            }
        ]
    }
)

registered_env_fixed = ml_client.environments.create_or_update(
    deployment_env_fixed
)

print("Fixed deployment environment registered successfully.")
print(f"Environment name: {registered_env_fixed.name}")
print(f"Environment version: {registered_env_fixed.version}")

Fixed deployment environment registered successfully.
Environment name: email-reply-distilbert-env
Environment version: 2
